In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np

# =========================
# PHASE 1: LOAD DATA
# =========================

retail = pd.read_excel("/content/drive/MyDrive/state_retail_data.xlsx")
unemployment = pd.read_excel("/content/drive/MyDrive/targetunemployment.xlsx")

# =========================
# PHASE 2: RETAIL WIDE → LONG
# =========================

retail_long = retail.melt(
    id_vars=["fips", "stateabbr", "naics"],
    var_name="DateRaw",
    value_name="Retail YoY Growth"
)

retail_long["Retail YoY Growth"] = pd.to_numeric(
    retail_long["Retail YoY Growth"],
    errors="coerce"
)

retail_long = retail_long.dropna(subset=["Retail YoY Growth"])

# =========================
# PHASE 3: STATE ABBR → FULL STATE
# =========================

state_map = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut",
    "DE": "Delaware", "DC": "District of Columbia", "FL": "Florida",
    "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois",
    "IN": "Indiana", "IA": "Iowa", "KS": "Kansas", "KY": "Kentucky",
    "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota",
    "MS": "Mississippi", "MO": "Missouri", "MT": "Montana",
    "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire",
    "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania",
    "RI": "Rhode Island", "SC": "South Carolina", "SD": "South Dakota",
    "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming"
}

retail_long["State"] = retail_long["stateabbr"].map(state_map)

# =========================
# PHASE 4: NAICS → SECTOR NAME
# =========================

naics_map = {
    441: "Motor Vehicle & Parts Dealers",
    442: "Furniture Stores",
    443: "Electronics Stores",
    444: "Building Materials Stores",
    445: "Food & Beverage Stores",
    446: "Health & Personal Care Stores",
    447: "Gasoline Stations",
    448: "Clothing Stores",
    451: "Sporting Goods / Hobby / Music",
    452: "General Merchandise Stores",
    453: "Miscellaneous Retailers",
    "TOTAL": "Total Retail"
}

retail_long["Sector"] = retail_long["naics"].map(naics_map)

# =========================
# PHASE 5: CLEAN UNEMPLOYMENT
# =========================

unemployment["Unemployment Rate"] = pd.to_numeric(
    unemployment["Unemployment Rate"],
    errors="coerce"
)

unemployment["Date"] = pd.to_datetime(
    unemployment["Date"],
    errors="coerce"
)

# =========================
# PHASE 6: JOIN DATASETS
# =========================

master = retail_long.merge(
    unemployment[["State", "DateRaw", "Date", "Unemployment Rate"]],
    on=["State", "DateRaw"],
    how="left"
)

# =========================
# PHASE 7: ADD ANALYSIS FIELDS
# =========================

master["Opportunity Score"] = (
    master["Retail YoY Growth"] - master["Unemployment Rate"]
)

master["Year"] = master["Date"].dt.year
master["Month"] = master["Date"].dt.month

# =========================
# PHASE 8: LEADING SECTOR BY STATE
# =========================

sector_only = master[master["naics"] != "TOTAL"].copy()

state_sector_avg = (
    sector_only
    .groupby(["State", "stateabbr", "fips", "naics", "Sector"], as_index=False)
    .agg({
        "Retail YoY Growth": "mean",
        "Unemployment Rate": "mean",
        "Opportunity Score": "mean"
    })
)

idx = state_sector_avg.groupby("State")["Retail YoY Growth"].idxmax()

leading_sector = state_sector_avg.loc[idx].copy()

leading_sector = leading_sector.rename(columns={
    "naics": "Leading NAICS",
    "Sector": "Leading Sector",
    "Retail YoY Growth": "Leading Sector Avg YoY Growth",
    "Opportunity Score": "Leading Sector Opportunity Score"
})

master = master.merge(
    leading_sector[
        [
            "State",
            "Leading NAICS",
            "Leading Sector",
            "Leading Sector Avg YoY Growth",
            "Leading Sector Opportunity Score"
        ]
    ],
    on="State",
    how="left"
)

master["Is Leading Sector"] = master["Sector"] == master["Leading Sector"]

# =========================
# PHASE 9: FINAL COLUMN NAMES
# =========================

final_df = master.rename(columns={
    "fips": "Fips",
    "stateabbr": "Stateabbr",
    "naics": "NAICS"
})

final_df = final_df[
    [
        "State",
        "Stateabbr",
        "Fips",
        "NAICS",
        "Sector",
        "DateRaw",
        "Date",
        "Year",
        "Month",
        "Retail YoY Growth",
        "Unemployment Rate",
        "Opportunity Score",
        "Leading NAICS",
        "Leading Sector",
        "Leading Sector Avg YoY Growth",
        "Leading Sector Opportunity Score",
        "Is Leading Sector"
    ]
]

# =========================
# PHASE 10: CHECK + EXPORT
# =========================

print("Final shape:", final_df.shape)
print("Missing unemployment:", final_df["Unemployment Rate"].isna().sum())
print("Missing state:", final_df["State"].isna().sum())
print("Missing sector:", final_df["Sector"].isna().sum())

display(final_df.head())

final_df.to_csv("final_retail_sector_state_analysis.csv", index=False)

Final shape: (21149, 17)
Missing unemployment: 605
Missing state: 0
Missing sector: 0


,State,Stateabbr,Fips,NAICS,Sector,DateRaw,Date,Year,Month,Retail YoY Growth,Unemployment Rate,Opportunity Score,Leading NAICS,Leading Sector,Leading Sector Avg YoY Growth,Leading Sector Opportunity Score,Is Leading Sector
0,Alabama,AL,1,441,Motor Vehicle & Parts Dealers,yy202301,2023-01-01,2023,1,2.0,2.3,-0.3,453,Miscellaneous Retailers,7.148571,4.485294,False
1,Alabama,AL,1,442,Furniture Stores,yy202301,2023-01-01,2023,1,11.8,2.3,9.5,453,Miscellaneous Retailers,7.148571,4.485294,False
2,Alabama,AL,1,443,Electronics Stores,yy202301,2023-01-01,2023,1,-3.3,2.3,-5.6,453,Miscellaneous Retailers,7.148571,4.485294,False
3,Alabama,AL,1,444,Building Materials Stores,yy202301,2023-01-01,2023,1,8.1,2.3,5.8,453,Miscellaneous Retailers,7.148571,4.485294,False
4,Alabama,AL,1,445,Food & Beverage Stores,yy202301,2023-01-01,2023,1,7.0,2.3,4.7,453,Miscellaneous Retailers,7.148571,4.485294,False
